# Face Model Lab: Video Bearbeitung & Face Blurring

Dieses Notebook nutzt die eigenständige Datei `blur_video.py`. Es ist für Smoke-Tests, Modellvergleich im Video und finale Blur-Läufe gedacht.

In [1]:
from pathlib import Path
import os
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd().parent / "matplotlib_cache"))
import subprocess
import sys

from IPython.display import Video, display

LAB = Path.cwd()
ROOT = LAB.parent
PYTHON = sys.executable
VIDEOS = ROOT / "Videos"
MODELS = ROOT / "trained_models"

print("Lab:", LAB)
print("Videos:", VIDEOS)
print("Python:", PYTHON)


Lab: /home/clemi/projekte/MIM/face_model_lab
Videos: /home/clemi/projekte/MIM/Videos
Python: /home/clemi/.venvs/MIM/bin/python


In [2]:
def run_cmd(args):
    print("$", " ".join(map(str, args)))
    result = subprocess.run(args, cwd=ROOT, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")


## Modell und Video auswählen

Nimm für schnelle Tests `face_yolov8m.pt`. Für eigene trainierte Modelle wähle ein `.pt`-Modell aus `trained_models/`, z.B. ein YOLO- oder RT-DETR-Finetuning.

In [3]:
model_candidates = [
    ROOT / "face_yolov8m.pt",
    *sorted(MODELS.glob("yolo*_bs*_ep*.pt")),
    *sorted(MODELS.glob("rtdetr*_bs*_ep*.pt")),
]
model_path = next((p for p in model_candidates if p.exists()), None)
input_video = VIDEOS / "Feuerwehr_progressiv.mp4"
output_video = VIDEOS / "Feuerwehr_lab_blur_notebook_smoke.mp4"

print("Model:", model_path if model_path else "kein Ultralytics-.pt-Modell gefunden")
print("Input:", input_video)
print("Output:", output_video)


Model: kein Ultralytics-.pt-Modell gefunden
Input: /home/clemi/projekte/MIM/Videos/Feuerwehr_progressiv.mp4
Output: /home/clemi/projekte/MIM/Videos/Feuerwehr_lab_blur_notebook_smoke.mp4


## Smoke-Test: wenige Frames blurren

In [4]:
if model_path is None:
    print("Überspringe Video-Smoke-Test: Für blur_video.py wird ein YOLO/RT-DETR .pt-Modell benötigt.")
else:
    run_cmd([
        PYTHON, "face_model_lab/blur_video.py",
        "--model", str(model_path),
        "--input", str(input_video),
        "--output", str(output_video),
        "--conf", "0.25",
        "--imgsz", "640",
        "--frame-skip", "2",
        "--max-frames", "30",
    ])


Überspringe Video-Smoke-Test: Für blur_video.py wird ein YOLO/RT-DETR .pt-Modell benötigt.


## Ausgabe anzeigen

In [5]:
if output_video.exists():
    display(Video(str(output_video), embed=False, width=720))
else:
    print("Output wurde nicht gefunden:", output_video)


Output wurde nicht gefunden: /home/clemi/projekte/MIM/Videos/Feuerwehr_lab_blur_notebook_smoke.mp4


## Finaler Lauf

Für ein komplettes Video `--max-frames` entfernen. Höheres `imgsz` findet kleine Gesichter besser, kostet aber mehr VRAM/Zeit.

In [6]:
# Beispiel für komplettes Rendern. Bei Bedarf ausführen.
# run_cmd([
#     PYTHON, "face_model_lab/blur_video.py",
#     "--model", str(model_path),
#     "--input", str(input_video),
#     "--output", str(VIDEOS / "Feuerwehr_lab_blur_full.mp4"),
#     "--frame-skip", "2",
#     "--imgsz", "960",
# ])
